# RGCA LIMIT=50 Real-Generation Input Builder for Kaggle

Run this notebook before the real-generation LIMIT=50 notebook. Its only goal is to create the private Kaggle input package needed by generation.

It performs:

- Repository setup
- MIMIC pilot subset detection or preparation
- Pilot JPG hydration from PhysioNet
- BioMedCLIP retrieval validation for 50 eval studies
- Strict row-count checks for retrieval and mismatch artifacts
- Packaging into a zip with the exact folder structure expected by the real-generation notebook

Expected final artifact:

```text
/kaggle/working/rgca_real_generation_input_limit50.zip
```

After it finishes, upload the zip contents as a new private Kaggle dataset and attach that dataset to the real-generation LIMIT=50 notebook.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

print('Python:', sys.version)
print('Kaggle working exists:', Path('/kaggle/working').exists())

# -----------------------------
# User-facing configuration
# -----------------------------
REPO_URL = 'https://github.com/pidoxy/RGCA.git'
PROJECT_ROOT = Path('/kaggle/working/RGCA')
WORK_DIR = Path('/kaggle/working/physionet')
HYDRATED_SUBSET_JSONL = Path('/kaggle/working/rgca_hydrated_subset/mimic_subset.jsonl')
BIOMEDCLIP_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/biomedclip_retrieval_validation_v0')
INPUT_PACKAGE_DIR = Path('/kaggle/working/rgca_real_generation_input_limit50')
INPUT_PACKAGE_ZIP = Path('/kaggle/working/rgca_real_generation_input_limit50.zip')

RETRIEVAL_LIMIT = 400
EVAL_LIMIT = 100
BIOMEDCLIP_EVAL_LIMIT = 50
TOP_K = 3

# This notebook is intentionally retrieval-input only. Do not rerun the old stress baseline here.
RUN_STRESS_BASELINE = False
RUN_BIOMEDCLIP_RETRIEVAL = True

# -----------------------------
# Helpers
# -----------------------------
def run(command, cwd=None, env=None):
    cwd = Path(cwd or PROJECT_ROOT)
    printable = []
    for part in command:
        text = str(part)
        if 'PHYSIONET' in text or len(text) > 140:
            printable.append(text[:140] + '...')
        else:
            printable.append(text)
    print('+', ' '.join(printable))
    return subprocess.run([str(part) for part in command], cwd=str(cwd), env=env, check=True)


def read_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def read_jsonl(path):
    path = Path(path)
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]


def find_subset_jsonl():
    candidates = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if root.exists():
            candidates.extend(root.rglob('mimic_subset.jsonl'))
    candidates = sorted(set(candidates))
    print('Subset candidates:')
    for path in candidates[:30]:
        print(' -', path)
    return candidates[0] if candidates else None


def ensure_repo():
    if PROJECT_ROOT.exists():
        print('Repo already exists:', PROJECT_ROOT)
        try:
            run(['git', 'pull'], cwd=PROJECT_ROOT)
        except Exception as exc:
            print('Repo pull failed; continuing with existing checkout:', exc)
    else:
        run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], cwd=Path('/kaggle/working'))
    run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_ROOT)
    if str(PROJECT_ROOT / 'src') not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT / 'src'))
    print('Repo ready:', PROJECT_ROOT)


def prepare_subset_from_physionet(username, password):
    if not username or not password:
        raise RuntimeError('No attached mimic_subset.jsonl found, and PhysioNet secrets are missing.')
    env = os.environ.copy()
    env['PHYSIONET_PASS'] = password
    output_dir = Path('/kaggle/working/rgca_pilot_500')
    run([
        sys.executable,
        'scripts/kaggle_prepare_mimic_subset.py',
        '--physionet-user', username,
        '--work-dir', WORK_DIR,
        '--output-dir', output_dir,
        '--retrieval-limit', RETRIEVAL_LIMIT,
        '--eval-limit', EVAL_LIMIT,
    ], cwd=PROJECT_ROOT, env=env)
    subset = output_dir / 'data' / 'mimic_subset.jsonl'
    if not subset.exists():
        raise FileNotFoundError(f'Expected prepared subset missing: {subset}')
    return subset


def hydrate_pilot_images(subset_path, username, password):
    if not username or not password:
        raise RuntimeError('BioMedCLIP retrieval needs PHYSIONET_USERNAME and PHYSIONET_PASS Kaggle secrets.')

    script = PROJECT_ROOT / 'scripts' / 'kaggle_hydrate_mimic_images.py'
    if script.exists():
        env = os.environ.copy()
        env['PHYSIONET_PASS'] = password
        run([
            sys.executable,
            script,
            '--subset', subset_path,
            '--physionet-user', username,
            '--output-root', '/kaggle/working/physionet/mimic-cxr-jpg/files',
            '--updated-subset', HYDRATED_SUBSET_JSONL,
        ], cwd=PROJECT_ROOT, env=env)
        return HYDRATED_SUBSET_JSONL

    raise RuntimeError('Missing scripts/kaggle_hydrate_mimic_images.py after repo setup. Run git pull or use the latest repo notebook.')


def validate_subset(subset_path, require_images=False):
    from rgca_baseline.integrity import jsonl_fingerprint, validate_image_paths, validate_study_records
    from rgca_baseline.pipeline import load_studies
    studies = load_studies(subset_path)
    validation = validate_study_records(studies)
    image_validation = validate_image_paths(studies)
    split_counts = {}
    for study in studies:
        split_counts[study.split] = split_counts.get(study.split, 0) + 1
    summary = {
        'subset': str(subset_path),
        'records': len(studies),
        'split_counts': split_counts,
        'fingerprint': jsonl_fingerprint(subset_path),
        'dataset_validation': validation,
        'image_validation': image_validation,
    }
    print(json.dumps(summary, indent=2)[:6000])
    if not validation['valid']:
        raise RuntimeError('Dataset contract validation failed.')
    if require_images and not image_validation['valid']:
        raise RuntimeError('Image validation failed after hydration; cannot build BioMedCLIP retrieval artifacts.')
    return summary


def assert_retrieval_artifacts(output_dir, expected_rows):
    output_dir = Path(output_dir)
    summary_path = output_dir / 'retrieval_validation_summary.json'
    retrieval_path = output_dir / 'retrieval_results.jsonl'
    mismatch_path = output_dir / 'mismatch_results.jsonl'
    for path in [summary_path, retrieval_path, mismatch_path]:
        if not path.exists():
            raise FileNotFoundError(f'Missing expected retrieval artifact: {path}')
    summary = json.loads(summary_path.read_text())
    retrieval_rows = read_jsonl(retrieval_path)
    mismatch_rows = read_jsonl(mismatch_path)
    checks = {
        'summary_eval_size': summary.get('eval_size'),
        'retrieval_rows': len(retrieval_rows),
        'mismatch_rows': len(mismatch_rows),
        'expected_rows': expected_rows,
        'top_k': summary.get('top_k'),
        'backend': summary.get('backend'),
    }
    print('Retrieval artifact checks:')
    print(json.dumps(checks, indent=2))
    if summary.get('backend') != 'biomedclip':
        raise RuntimeError(f'Expected backend biomedclip, got {summary.get("backend")}')
    if summary.get('top_k') != TOP_K:
        raise RuntimeError(f'Expected top_k={TOP_K}, got {summary.get("top_k")}')
    if summary.get('eval_size') != expected_rows:
        raise RuntimeError(f'Expected summary eval_size={expected_rows}, got {summary.get("eval_size")}')
    if len(retrieval_rows) != expected_rows or len(mismatch_rows) != expected_rows:
        raise RuntimeError(f'Expected {expected_rows} retrieval/mismatch rows, got {len(retrieval_rows)} and {len(mismatch_rows)}')
    return summary


def package_real_generation_input():
    if INPUT_PACKAGE_DIR.exists():
        shutil.rmtree(INPUT_PACKAGE_DIR)
    if INPUT_PACKAGE_ZIP.exists():
        INPUT_PACKAGE_ZIP.unlink()

    hydrated_dst = INPUT_PACKAGE_DIR / 'rgca_hydrated_subset'
    retrieval_dst = INPUT_PACKAGE_DIR / 'rgca_experiments' / 'biomedclip_retrieval_validation_v0'
    hydrated_dst.mkdir(parents=True, exist_ok=True)
    retrieval_dst.mkdir(parents=True, exist_ok=True)

    shutil.copy2(HYDRATED_SUBSET_JSONL, hydrated_dst / 'mimic_subset.jsonl')
    manifest = HYDRATED_SUBSET_JSONL.parent / 'image_hydration_manifest.json'
    if manifest.exists():
        shutil.copy2(manifest, hydrated_dst / 'image_hydration_manifest.json')

    for name in ['retrieval_results.jsonl', 'mismatch_results.jsonl', 'retrieval_validation_summary.json']:
        shutil.copy2(BIOMEDCLIP_OUTPUT_DIR / name, retrieval_dst / name)

    package_manifest = {
        'purpose': 'Input package for RGCA real-generation LIMIT=50 notebook',
        'expected_generation_limit': BIOMEDCLIP_EVAL_LIMIT,
        'top_k': TOP_K,
        'retrieval_backend': 'biomedclip',
        'required_files': [
            'rgca_hydrated_subset/mimic_subset.jsonl',
            'rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_results.jsonl',
            'rgca_experiments/biomedclip_retrieval_validation_v0/mismatch_results.jsonl',
        ],
    }
    (INPUT_PACKAGE_DIR / 'INPUT_PACKAGE_MANIFEST.json').write_text(json.dumps(package_manifest, indent=2))

    shutil.make_archive(str(INPUT_PACKAGE_ZIP.with_suffix('')), 'zip', str(INPUT_PACKAGE_DIR))
    print('input_package_dir:', INPUT_PACKAGE_DIR)
    print('input_package_zip:', INPUT_PACKAGE_ZIP, 'exists=', INPUT_PACKAGE_ZIP.exists(), 'size_mb=', round(INPUT_PACKAGE_ZIP.stat().st_size / (1024 * 1024), 3))
    return INPUT_PACKAGE_ZIP

# -----------------------------
# Execute LIMIT=50 input build
# -----------------------------
ensure_repo()
physionet_username = read_secret('PHYSIONET_USERNAME') or read_secret('PHYSIONET_USER')
physionet_password = read_secret('PHYSIONET_PASS') or read_secret('PHYSIONET_PASSWORD')
print('PHYSIONET_USERNAME set:', bool(physionet_username))
print('PHYSIONET_PASS set:', bool(physionet_password))

subset = find_subset_jsonl()
if subset is None:
    subset = prepare_subset_from_physionet(physionet_username, physionet_password)
print('Using subset:', subset)
# The raw attached subset may contain stale /kaggle/working image paths from an older session.
# Before hydration, validate only the study/report contract. Real image paths are checked after hydration.
validate_subset(subset, require_images=False)

if not RUN_BIOMEDCLIP_RETRIEVAL:
    raise RuntimeError('RUN_BIOMEDCLIP_RETRIEVAL must stay True for this notebook.')

hydrated_subset = hydrate_pilot_images(subset, physionet_username, physionet_password)
validate_subset(hydrated_subset, require_images=True)

run([sys.executable, '-m', 'pip', 'install', '-q', 'open_clip_torch', 'pillow'], cwd=PROJECT_ROOT)
if BIOMEDCLIP_OUTPUT_DIR.exists():
    shutil.rmtree(BIOMEDCLIP_OUTPUT_DIR)
run([
    sys.executable,
    'scripts/run_retrieval_validation.py',
    '--subset', hydrated_subset,
    '--backend', 'biomedclip',
    '--output-dir', BIOMEDCLIP_OUTPUT_DIR,
    '--top-k', TOP_K,
    '--eval-limit', BIOMEDCLIP_EVAL_LIMIT,
], cwd=PROJECT_ROOT)

retrieval_summary = assert_retrieval_artifacts(BIOMEDCLIP_OUTPUT_DIR, BIOMEDCLIP_EVAL_LIMIT)
zip_path = package_real_generation_input()

print('\nLIMIT=50 input package is ready.')
print('Download from Kaggle Output panel:')
print(zip_path)
print('\nThen create/update a private Kaggle dataset from the zip contents and attach it to the real-generation LIMIT=50 notebook.')
print('\nCompact summary:')
print(json.dumps({
    'zip': str(zip_path),
    'hydrated_subset': str(HYDRATED_SUBSET_JSONL),
    'retrieval_results': str(BIOMEDCLIP_OUTPUT_DIR / 'retrieval_results.jsonl'),
    'mismatch_results': str(BIOMEDCLIP_OUTPUT_DIR / 'mismatch_results.jsonl'),
    'eval_size': retrieval_summary.get('eval_size'),
    'retrieval_pool_size': retrieval_summary.get('retrieval_pool_size'),
    'top_k': retrieval_summary.get('top_k'),
}, indent=2))



## Inspect Evidence Files

Run this after the main pipeline if you want a readable inventory of the generated outputs.


In [ ]:
from pathlib import Path
import json

paths = [
    Path('/kaggle/working/rgca_real_generation_input_limit50.zip'),
    Path('/kaggle/working/rgca_real_generation_input_limit50/INPUT_PACKAGE_MANIFEST.json'),
    Path('/kaggle/working/rgca_real_generation_input_limit50/rgca_hydrated_subset/mimic_subset.jsonl'),
    Path('/kaggle/working/rgca_real_generation_input_limit50/rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_results.jsonl'),
    Path('/kaggle/working/rgca_real_generation_input_limit50/rgca_experiments/biomedclip_retrieval_validation_v0/mismatch_results.jsonl'),
]
for path in paths:
    print(path, 'exists=', path.exists(), 'size_mb=', round(path.stat().st_size/(1024*1024), 3) if path.exists() else None)

summary = Path('/kaggle/working/rgca_real_generation_input_limit50/rgca_experiments/biomedclip_retrieval_validation_v0/retrieval_validation_summary.json')
if summary.exists():
    data = json.loads(summary.read_text())
    print(json.dumps({
        'backend': data.get('backend'),
        'top_k': data.get('top_k'),
        'retrieval_pool_size': data.get('retrieval_pool_size'),
        'eval_size': data.get('eval_size'),
        'outputs': data.get('outputs'),
    }, indent=2))
